# Script para envio de mensagens simples  

_Premisas:  _

1. Carregar bibliotecas necessarias
    1. Pandas
    2. Selenium
    3. Webdriver Manager
    4. Time
2. Carregar arquivo de mensagens en dataframe
3. Limpar dataframe  
    1. Eliminar linhas sem mensagens
    2. Eliinar linhas sem telefones
    3. Eliminar colunas desnecessarias
    4. Obter o total de linhas
    5. Resetar indice
4. Abrir o Navegador
5. Abrir o WhatsappWeb
6. Esperar conexão do celular
7. Iniciar Loop de envio de imagens
    1. Enviar mensagem
    2. Iniciar Loop de envio de imagens

Carregar bibliotecas necessarias

In [ ]:
import pandas as pd

# necessary libraries for Chrome operations:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import NoSuchElementException

# modified 29-oct-23
from selenium.webdriver.chrome.service import Service

# pip install webdriver_manager
# This librari updates automatically the Browser Manger (in this case, Chrome)
from webdriver_manager.chrome import ChromeDriverManager

# Necessary to convert messages from ASCII text into URL aceptable addresses (convert special characters, spaces, etc)
import urllib

# Just to get image file name from full path
from pathlib import Path

# Time to allow program wait few seconds during Chrome operations
import time

# To allow randomic waiting times (important to avoid Whatsapp account blocking)
import random

# Datetime to store current date of messages sent
# from datetime import date
import datetime as dt

# importar Tkinter
import tkinter as tk
from tkinter import filedialog as fd

In [ ]:
# 1 de março
# vamos tentar com import paperclip
import pyperclip
import pyautogui

Carregar biblioteca de obejtos clicaveis

In [ ]:
import whatsappweb_objects as wwo

In [ ]:
# isto é apenas para recarregar a biblioteca
import importlib

Carregar arquivo de mensagens en dataframe

In [ ]:
arquivo_de_mensagens = fd.askopenfilename(
    title='Selecione o arquivo Excel com a lista de destinatarios',
    filetypes=[('Arquivo Excel','.xls'),('Arquivo Excel','.xlsx')]
    )

contacts_df = pd.read_excel(arquivo_de_mensagens, sheet_name='CLIENTES')

In [ ]:
contacts_df

Carregar lista de imagens

In [ ]:
# selecionar arquivos de imagen

images_types = [
        ('Arquivos de imagen','.jpg'),
        ('Arquivos de imagen','.jpeg'),
        ('Arquivos de imagen','.png'),
        ('Arquivos de imagen','.gif'),
        ]
 
imgs_path = sorted(list(fd.askopenfilenames(title='Selecione as imagens a enviar',filetypes=images_types)))

In [ ]:
imgs_path

Limpar dataframe

In [ ]:
# 1. Eliminar linhas sem mensagens
contacts_df = contacts_df[~contacts_df['MENSAGEM'].isnull()]

# 2. Eliinar linhas sem telefones
contacts_df = contacts_df[~contacts_df['TELEFONE'].isnull()]

# 3. Reset index
contacts_df.reset_index(inplace=True)

# 4. Eliminar colunas desnecessarias
contacts_df = contacts_df[['CLIENTE','TELEFONE','MENSAGEM']]

# 5. Obter o total de linhas
numero_de_mensagens = contacts_df['MENSAGEM'].count()

print('Serão enviadas {} mensagens'.format(numero_de_mensagens))

In [ ]:
# visualizar dataframe
contacts_df

Abrir o Navegador

In [ ]:
# Criar uma instancia do Google Chrome
msg_browser = webdriver.Chrome()

Abrir o WhatsappWeb

In [ ]:
# Navegar até o WhatsApp Web
msg_browser.get("https://web.whatsapp.com/")
time.sleep(2)

Esperar conexão do celular

In [ ]:
# Esperar pela lista de contatos do WhatsApp por X segundos
# Isto indica que podemos começar a enviar mensagens
while len(msg_browser.find_elements(By.ID,"side")) < 1:
    time.sleep(1)

# Aqui começa o teste

In [ ]:
# en caso de modificar a biblioteca, recarregar
importlib.reload(wwo)

# ESTE FUNCIONA

So não detectou o telefone errado. verificar se é problema de objeto

In [ ]:
for j, mensagem in enumerate(contacts_df['MENSAGEM']):
    # pegar a linha j
    mensagem = contacts_df.loc[j,'MENSAGEM']
    cliente = contacts_df.loc[j,'CLIENTE']
    telefone = contacts_df.loc[j,'TELEFONE']
    print(j, cliente, telefone)
    # Converter a mensagem de ASCII para texto plano para ser usada como URL
    url_mensagem = urllib.parse.quote(f"{mensagem}")

    # Construir o link
    link = f"https://web.whatsapp.com/send?phone={telefone}&text={url_mensagem}"
    # link = f"https://wa.me/{telefone}?text={url_mensagem}"
    msg_browser.get(link)
    # verificar se pode ser enviado ou se for um numero errado
    telefone_errado = False
    envio_disponivel = False

    while (not(telefone_errado) and not(envio_disponivel)):
        # capturar se aparece o botão de envio
        try:
            time.sleep(1)
            msg_browser.find_element(By.CSS_SELECTOR,wwo.objects['send_btn'])
            envio_disponivel = True
            # print("pegou envio disponivel")
            break   # SE JA SE QUE ENCONTREI O BOTAO DE ENVIO, NAO FAZ SENTIDO ESPERAR PELO TELEFONE ERRADO
        except NoSuchElementException:
            envio_disponivel = False
            # print("nao encontrou enviar")
        
        # capturar telefone errado
        try:
            time.sleep(1)
            msg_browser.find_element(By.CSS_SELECTOR,wwo.objects['wrong_number_wrng'])
            # msg_browser.find_element(By.CSS_SELECTOR,wrong_number_continue_btn)
            telefone_errado = True
            # print("pegou telefone errado")
            break   # SE JA SEI QUE O TELEFONE ESTA ERRADO, NAO FAZ SENTIDO ESPERAR PELO ENVIO
        except NoSuchElementException:
            telefone_errado = False
            # print("nao e telefone errado")
    
    # print("Telefone errado: ", telefone_errado)
    # print("Envio disponivel: ", envio_disponivel)
    # modificações 27-set-2025
    # se aparece botao de envio, clicar nele
    if envio_disponivel:
        msg_browser.find_element(By.CSS_SELECTOR,wwo.objects['send_btn']).click()
    # se aparece telefone errado, clicar nele
    if telefone_errado:
        msg_browser.find_element(By.CSS_SELECTOR,wwo.objects['wrong_number_continue_btn']).click()
    time.sleep(2)
    ### Aqui começa a enviar imagens
    for i, img_file in enumerate(imgs_path):

        # img_file = imgs_path[i]
        # print(img_file.replace("/", "\\"))
        # wwo.objects['pictures_and_videos_btn'] = '//*[@id="app"]/div[1]/div/span[6]/div/ul/div/div/div[2]/li/div/input'
        wwo.objects['pictures_and_videos_btn'] = '#app > div > div > div:nth-child(11) > div > div > div.xu96u03.xm80bdy.x10l6tqk.x13vifvy.xoz0ns6.x1gslohp > div.html-div.xdj266r.x14z9mp.xat24cr.x1lziwak.xexx8yu.xyri2b.x18d9i69.x1c1uobl > div > div > div > div > div.x78zum5.xdt5ytf.x1iyjqo2.x1n2onr6 > div:nth-child(2)'

        pyperclip.copy(img_file.replace("/", "\\"))
        # click on add attachment (plus sign)
        msg_browser.find_element(By.CSS_SELECTOR,wwo.objects['add_attachment_btn']).click()
        time.sleep(2)
        # clicar em "fotos e videos"
        # o xpath tem que ter a palavra "input"
        # Gemint passou usar um conector diferente, ver no diccionario
        msg_browser.find_element(By.CSS_SELECTOR, wwo.objects['pictures_and_videos_btn']).click()
        # msg_browser.find_element(By.XPATH, wwo.objects['pictures_and_videos_btn']).send_keys(img_file)
        time.sleep(2)
        # 4. Pequena pausa para você clicar na janela ou para o sistema processar
        # print("Você tem 3 segundos para focar no campo de texto...")
        time.sleep(3)

        # 5. Simular o Colar (Ctrl+V) e o Enter
        pyautogui.hotkey('ctrl', 'v')
        pyautogui.press('enter')
        time.sleep(3)
        # clicar no triangulo de envio
        msg_browser.find_element(By.CSS_SELECTOR,wwo.objects['send_attachment_btn']).click()
        time.sleep(3)
        # lembra de adicionar tempo aqui antes de enviar a proxima mensagem